# 06 — Real day-ahead CAMS forecasts (ADS download)

**Why:** In A1-revision experiment E2 we used CAMS values from the Open-Meteo archive, which keeps the *latest* run for each hour. That is slightly optimistic for a real "tomorrow" forecast. Here we download the forecasts that were **actually available the day before** from the Copernicus Atmosphere Data Store (ADS).

**What:** CAMS European ensemble (median of 11 models), surface level, run 00 UTC of day *D*, lead times **21–44 h** → this covers exactly the local (Europe/Istanbul, UTC+3) calendar day *D+1*, 00:00–23:00. The run is published ~06:45 UTC on day *D*, so it is known before day *D* ends (our decision time).

Variables: PM10, PM2.5, NO2, SO2, dust. Area: small box around the Arnavutköy station (41.2209 N, 28.7075 E).

**Output:** `cams_dayahead_hourly.csv` and `cams_dayahead_daily.csv` (local-day mean and median, ≥18 valid hours) in the Drive folder below. Copy them to `data/processed/` of the repo.

**How to run:** Runtime → Run all. Paste your ADS API key when asked (it is not saved in the notebook). Jobs wait in the ADS queue; leave the tab open. If Colab disconnects, just run all again: finished months are skipped and submitted jobs are resumed.

In [ ]:
!pip -q install ecmwf-datastores-client xarray netCDF4 h5netcdf

In [ ]:
import os, json, time, zipfile, glob, re
from datetime import date, timedelta
import numpy as np, pandas as pd, xarray as xr
from getpass import getpass
from ecmwf.datastores import Client

# Keep files on Google Drive so nothing is lost if Colab disconnects
from google.colab import drive
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/arnavutkoy_cams_dayahead'
RAW = f'{OUT}/raw'
os.makedirs(RAW, exist_ok=True)

STATION_LAT, STATION_LON = 41.220937, 28.707547
AREA = [41.5, 28.4, 40.9, 29.0]          # N, W, S, E
DATASET = 'cams-europe-air-quality-forecasts'
VARIABLES = ['particulate_matter_10um', 'particulate_matter_2.5um',
             'nitrogen_dioxide', 'sulphur_dioxide', 'dust']
LEADS = [str(h) for h in range(21, 45)]  # 21..44 h -> local day D+1

KEY = getpass('ADS API key: ')
client = Client(url='https://ads.atmosphere.copernicus.eu/api', key=KEY, progress=False)
client.check_authentication()
print('Authentication OK')

## 1. Request definition and a quick validity check
The request is checked with ADS's cost estimate before anything is submitted. If the `netcdf_zip` format is not accepted, the cell tries the alternatives automatically.

In [ ]:
def make_request(d0, d1, fmt):
    return {
        'variable': VARIABLES,
        'model': ['ensemble'],
        'level': ['0'],
        'type': ['forecast'],
        'time': ['00:00'],
        'leadtime_hour': LEADS,
        'date': [f'{d0:%Y-%m-%d}/{d1:%Y-%m-%d}'],
        'data_format': fmt,
        'area': AREA,
    }

FMT = None
for fmt in ['netcdf_zip', 'netcdf']:
    try:
        est = client.estimate_costs(DATASET, make_request(date(2025, 7, 1), date(2025, 7, 31), fmt))
        print(fmt, '->', est)
        FMT = fmt
        break
    except Exception as e:
        print(fmt, 'not accepted:', str(e)[:300])
assert FMT, 'No accepted format - send this output to Claude.'
print('Using format:', FMT)

## 2. Monthly chunks
Run dates cover the **validation + test** periods first (07.07.2025 – 16.09.2026 → run dates 06.07.2025 – 15.09.2026), then the **training** period (23.09.2023 – 06.07.2025), so that E2 can also be *trained* on real day-ahead forecasts. ADS keeps a 3-year rolling archive, so the first training months may be unavailable; those are simply reported and skipped.

If a month is above the ADS size limit it is split in two automatically.

In [ ]:
def month_chunks(start, end):
    out, d = [], start
    while d <= end:
        nxt = (d.replace(day=1) + timedelta(days=32)).replace(day=1)
        out.append((d, min(nxt - timedelta(days=1), end)))
        d = nxt
    return out

def fits(d0, d1):
    try:
        est = client.estimate_costs(DATASET, make_request(d0, d1, FMT))
        cost, limit = est.get('cost'), est.get('limit')
        if isinstance(cost, (int, float)) and isinstance(limit, (int, float)):
            return cost <= limit
    except Exception as e:
        print('estimate failed', d0, d1, str(e)[:200])
    return True

def split_to_fit(d0, d1):
    if d0 == d1 or fits(d0, d1):
        return [(d0, d1)]
    mid = d0 + (d1 - d0) // 2
    return split_to_fit(d0, mid) + split_to_fit(mid + timedelta(days=1), d1)

priority = month_chunks(date(2025, 7, 6), date(2026, 9, 15))   # validation + test
training = month_chunks(date(2023, 9, 22), date(2025, 7, 5))   # training
CHUNKS = []
for a, b in priority + training:
    CHUNKS += split_to_fit(a, b)
print(len(CHUNKS), 'requests')

## 3. Submit, wait, download
ADS accepts only **one queued request per dataset per user**, so months are requested one after another (validation + test first). Job ids are saved to Drive, so a re-run resumes instead of submitting again.

In [ ]:
# ADS allows only ONE queued request per dataset per user -> strictly sequential.
JOBS = f'{OUT}/jobs.json'
jobs = json.load(open(JOBS)) if os.path.exists(JOBS) else {}
def save_jobs(): json.dump(jobs, open(JOBS, 'w'), indent=1)

def target(d0, d1):
    ext = 'zip' if FMT == 'netcdf_zip' else 'nc'
    return f'{RAW}/cams_{d0:%Y%m%d}_{d1:%Y%m%d}.{ext}'

def get_chunk(a, b):
    k = f'{a}_{b}'
    while True:
        if k not in jobs:
            try:
                jobs[k] = client.submit(DATASET, make_request(a, b, FMT)).request_id; save_jobs()
            except Exception as e:
                print('  submit refused, retry in 60 s:', str(e)[:150]); time.sleep(60); continue
        try:
            r = client.get_remote(jobs[k])
            while not r.results_ready:
                time.sleep(20)
            r.download(target(a, b))
            return 'ok'
        except Exception as e:
            msg = str(e)
            del jobs[k]; save_jobs()               # forget the dead job id
            if 'queued requests' in msg or 'Maximum number' in msg:
                print('  queue busy, retry in 60 s'); time.sleep(60); continue
            return msg[:300]                      # real failure (e.g. outside 3-year archive)

failed = {}
todo = [(a, b) for a, b in CHUNKS if not os.path.exists(target(a, b))]
print(len(todo), 'chunks to download')
t0 = time.time()
for i, (a, b) in enumerate(todo, 1):
    print(f'{time.strftime("%H:%M")}  [{i}/{len(todo)}] {a} -> {b}')
    res = get_chunk(a, b)
    if res == 'ok':
        print(f'  downloaded ({(time.time()-t0)/60:.0f} min so far)')
    else:
        print('  FAILED:', res); failed[f'{a}_{b}'] = res
print('Done. Failed chunks:', failed if failed else 'none')

## 4. Extract the station point and build hourly / daily tables
The station is matched to the nearest 0.1° grid point. Valid time = run time + lead time, converted to Europe/Istanbul.

In [ ]:
NAME_MAP = [('dust', 'dust'), ('pm2p5', 'pm25'), ('pm2.5', 'pm25'), ('pm10', 'pm10'),
            ('no2', 'no2'), ('nitrogen_dioxide', 'no2'), ('so2', 'so2'), ('sulphur', 'so2')]
def short(v):
    v = v.lower()
    for key, s in NAME_MAP:
        if key in v:
            return s
    return None

def coord(ds, *names):
    for n in names:
        if n in ds.coords or n in ds.dims:
            return n
    return None

def base_from(ds, path):
    txt = ' '.join(str(x) for x in [ds[c].attrs for c in ds.coords] + [ds.attrs, os.path.basename(path)])
    m = re.search(r'(20\d{6})', txt)
    return pd.Timestamp(m.group(1)) if m else None

def nc_to_df(ds, path):
    lat, lon = coord(ds, 'latitude', 'lat'), coord(ds, 'longitude', 'lon')
    pt = ds.sel({lat: STATION_LAT, lon: STATION_LON % 360 if float(ds[lon].max()) > 180 else STATION_LON},
                method='nearest')
    keep = {v: short(v) for v in pt.data_vars if short(v)}
    df = pt[list(keep)].to_dataframe().reset_index().rename(columns=keep)
    if 'valid_time' in df:
        vt = pd.to_datetime(df['valid_time'])
    elif 'forecast_reference_time' in df and ('forecast_period' in df or 'step' in df):
        step = df['forecast_period'] if 'forecast_period' in df else df['step']
        vt = pd.to_datetime(df['forecast_reference_time']) + pd.to_timedelta(step)
    elif 'time' in df and 'step' in df:
        vt = pd.to_datetime(df['time']) + pd.to_timedelta(df['step'])
    elif 'time' in df and np.issubdtype(df['time'].dtype, np.timedelta64):
        vt = base_from(ds, path) + pd.to_timedelta(df['time'])
    else:
        vt = pd.to_datetime(df['time'])
    out = df[[c for c in ['pm10', 'pm25', 'no2', 'so2', 'dust'] if c in df]].copy()
    out.insert(0, 'valid_utc', vt.values)
    return out

def open_all(path):
    frames = []
    if path.endswith('.zip'):
        tmp = path[:-4] + '_unz'
        if not os.path.isdir(tmp):
            with zipfile.ZipFile(path) as z: z.extractall(tmp)
        files = sorted(glob.glob(f'{tmp}/**/*.nc', recursive=True))
    else:
        files = [path]
    for f in files:
        with xr.open_dataset(f) as ds:
            frames.append(nc_to_df(ds.load(), f))
    return frames

# Show the structure of one file (useful if anything goes wrong)
first = sorted(glob.glob(f'{RAW}/*.zip') + glob.glob(f'{RAW}/*.nc'))[0]
print(first)
if first.endswith('.zip'):
    with zipfile.ZipFile(first) as z: print(z.namelist()[:5])

In [ ]:
frames = []
for p in sorted(glob.glob(f'{RAW}/*.zip') + glob.glob(f'{RAW}/*.nc')):
    frames += open_all(p)
hourly = (pd.concat(frames).dropna(subset=['valid_utc'])
          .drop_duplicates('valid_utc').sort_values('valid_utc'))
hourly['time_local'] = (pd.to_datetime(hourly['valid_utc']).dt.tz_localize('UTC')
                        .dt.tz_convert('Europe/Istanbul').dt.tz_localize(None))
hourly['date'] = hourly['time_local'].dt.normalize()
pols = [c for c in ['pm10', 'pm25', 'no2', 'so2', 'dust'] if c in hourly]

# Sanity check: µg/m3 expected (PM10 typically 5-100). GRIB-style kg/m3 would be ~1e-8.
print(hourly[pols].describe().round(2))
if hourly['pm10'].median() < 1e-3:
    hourly[pols] = hourly[pols] * 1e9
    print('Converted kg/m3 -> µg/m3')

g = hourly.groupby('date')
n = g['pm10'].count()
daily = pd.concat({f'cams_da_{p}_{s}': getattr(g[p], s)() for p in pols for s in ['mean', 'median']}, axis=1)
daily = daily[n >= 18]
daily.index.name = 'date'
hourly.drop(columns='date').to_csv(f'{OUT}/cams_dayahead_hourly.csv', index=False)
daily.to_csv(f'{OUT}/cams_dayahead_daily.csv')
print(len(hourly), 'hours,', len(daily), 'days:', daily.index.min().date(), '->', daily.index.max().date())
daily.head()

## 5. Quick look: how different is the real day-ahead forecast from the archive?
If the archive file from notebook 03 is in the same Drive folder (or uploaded to `/content`), this cell compares them. Otherwise skip it — notebook 07 does the full comparison.

In [ ]:
cands = glob.glob('/content/**/*daily*.csv', recursive=True) + glob.glob(f'{OUT}/*daily*.csv')
cands = [c for c in cands if 'dayahead' not in c]
print('archive candidates:', cands)
if cands:
    arc = pd.read_csv(cands[0], parse_dates=[0], index_col=0)
    print('archive columns:', [c for c in arc.columns if 'cams' in c.lower()])